# BrainDecoding — Google Colab GPU 秒级解包与混合精度全量训练工作流

本 Notebook 运行在 **Google Colab GPU 实例**（如 T4 / L4 / A100）上：
1. 从 Google Drive 秒级解包单文件缓存归档 `archives/features_<modality>.tar` 到本地高速盘 `/content/local_cache`；
2. 在本地 `/content/studies_links` 建立与 CPU 预热阶段完全一致的软链接桥（确保缓存签名 100% 命中）；
3. 启动带 `--require-cache`、`--resume auto` 与 `--precision bf16-mixed` 的正式训练，所有 Checkpoint (`last.ckpt` / `best.ckpt`) 与 `subjects.json` 实时落盘至 Google Drive `runs/` 下的受试者子集专属目录。

In [ ]:
# 1. 检查 GPU 并挂载 Google Drive、配置环境变量
!nvidia-smi

import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 用户可修改参数 ====================
GIT_REPO_URL = 'https://github.com/YOUR_USERNAME/BrainDecoding.git'  # 替换为你的 Git 仓库地址
GIT_BRANCH = 'main'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'
DATASET = 'chineseeeg2_littleprince'  # 可选: 'chineseeeg2_littleprince' 或 'pallier2025'
MODALITY = 'eeg'                      # 对应归档标识: 'eeg' 或 'meg'
STAGE = 'word'                        # 训练阶段: 'word' 或 'revision'
SEED = 0                              # 随机种子: 0, 1, 2
PRECISION = 'bf16-mixed'              # 混合精度: 'bf16-mixed' 或 'fp16-mixed'
SUBSET_ARGS = '--subject all'         # 或 '--num-subjects 2' / '--subjects sub-01,sub-02'
# ========================================================

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'

for folder in (LINKS_DIR, LOCAL_CACHE_DIR, DRIVE_ARCHIVE_DIR, DRIVE_RESULTS_DIR):
    Path(folder).mkdir(parents=True, exist_ok=True)

# 统一通过环境变量驱动所有路径（与 CPU 预热完全对齐）
os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_MODEL_DIR'] = f'{DRIVE_BASE_DIR}/models'

print('BRAINDECODING_DATA_DIR    =', os.environ['BRAINDECODING_DATA_DIR'])
print('BRAINDECODING_CACHE_DIR   =', os.environ['BRAINDECODING_CACHE_DIR'])
print('BRAINDECODING_RESULTS_DIR =', os.environ['BRAINDECODING_RESULTS_DIR'])

In [ ]:
# 2. 克隆或同步最新代码到 /content/code
!if [ ! -d "{CODE_DIR}/.git" ]; then \
    git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}; \
else \
    git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only; \
fi

In [ ]:
# 3. 防崩版初始化虚拟环境（锁定系统 PyTorch/CUDA 版本）+ 建立 /content/studies_links 软链接桥
!python3 -u {CODE_DIR}/scripts/colab_setup.py \
    --venv-dir {VENV_DIR} \
    --project-root {CODE_DIR} \
    --drive-data-dir {DRIVE_DATA_DIR} \
    --links-dir {LINKS_DIR} \
    --local-cache-dir {LOCAL_CACHE_DIR}

In [ ]:
# 4. 将 Google Drive 上的单一 .tar 缓存包秒级解包回 /content/local_cache
!cd {CODE_DIR} && {PYTHON} -u scripts/cache_archive.py unpack \
    --archive-dir {DRIVE_ARCHIVE_DIR} \
    --cache-dir {LOCAL_CACHE_DIR} \
    --modality {MODALITY} \
    --skip-existing

In [ ]:
# 5. 启动 GPU 训练（强制 --require-cache 零现场重算 + --resume auto 自动从 last.ckpt 续训）
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding train \
    --dataset {DATASET} \
    --stage {STAGE} \
    --seed {SEED} \
    --precision {PRECISION} \
    --require-cache \
    --resume auto \
    {SUBSET_ARGS}